**V1 experiment.** This notebook contains the V1 implementation and experiment configuration. See [methods and evaluation](../docs/METHODS.md) for the implemented protocol.


# Level 3 — Spinner Dolphin Vocalization-Type Discovery (unsupervised)

**Goal:** Discover natural vocalization types (whistles, clicks, burst pulses)
in Spinner Dolphin recordings without labels.

Pipeline:
1. Load `Level3/Data/level3_features.npz`, standardize features.
2. Build k-NN graph (k=7, σ=median).
3. Spectral gap analysis: first 10 eigenvalues of L_sym.
4. Spectral clustering for k=2..8, silhouette score for each.
5. Diffusion map embedding colored by (i) cluster assignment, (ii) spectral flatness.
6. t-SNE comparison.
7. Validation: spectral-flatness proxy labels → ARI / NMI / spectrogram grids.
8. Autoencoder (46 → 32 → 16 → 32 → 46) + k-means baseline.


In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.graph import (
    build_knn_graph,
    spectral_gap_from_W,
    diffusion_map_embedding,
    spectral_clustering,
)
from utils.evaluation import plot_spectral_gap, plot_silhouette_vs_k

LEVEL_DIR = PROJECT_ROOT / "Level3"
DATA_DIR = LEVEL_DIR / "Data"
FIG_DIR = LEVEL_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("level dir:", LEVEL_DIR)


## Load and standardize features


In [ ]:
data = np.load(DATA_DIR / "level3_features.npz", allow_pickle=True)
X = data["X"].astype(np.float32)
clip_ids = data["clip_ids"]
print("X:", X.shape)

scaler = StandardScaler()
Xz = scaler.fit_transform(X).astype(np.float32)

# Spectral flatness was the 4th of the 4 spectral stats (mean) — column 30 in
# the 46-dim Level 3 vector (13 MFCC + 13 dMFCC + centroid + bandwidth + rolloff
# + flatness_mean + ...). flatness_std is the LAST column (index 45).
FLATNESS_MEAN_IDX = 13 + 13 + 3  # = 29 (centroid=26, bandwidth=27, rolloff=28, flatness=29)
flatness_mean = X[:, FLATNESS_MEAN_IDX]
print(f"spectral flatness range: {flatness_mean.min():.4f} -- {flatness_mean.max():.4f}")


## k-NN graph + spectral gap


In [ ]:
K_GRAPH = 7
W = build_knn_graph(Xz, k=K_GRAPH, sigma="median")
print(f"W: {W.shape}, nnz={W.nnz}")

eigs = spectral_gap_from_W(W, n_eigenvalues=10)
print("first 10 eigenvalues of L_sym:", eigs)
plot_spectral_gap(eigs, FIG_DIR / "spectral_gap.png", title="Level 3 — Spectral gap")

gap_diffs = np.diff(eigs)
suggested_k = int(1 + np.argmax(gap_diffs[1:])) + 1  # skip the trivial gap at 0
print(f"largest gap suggests k* = {suggested_k}")


## Spectral clustering for k = 2..8


In [ ]:
K_RANGE = list(range(2, 9))
silhouettes = []
clusterings = {}
for k in K_RANGE:
    cl = spectral_clustering(W, k=k, random_state=0)
    clusterings[k] = cl
    if len(set(cl)) > 1:
        s = silhouette_score(Xz, cl)
    else:
        s = float("nan")
    silhouettes.append(s)
    print(f"  k={k}: silhouette = {s:.4f}")

plot_silhouette_vs_k(silhouettes, K_RANGE, FIG_DIR / "silhouette_vs_k.png")
best_k_data = K_RANGE[int(np.nanargmax(silhouettes))]
print(f"\nbest k from silhouette = {best_k_data}")


## Diffusion map embedding

Colored by (a) cluster assignment at k=k*, and (b) raw spectral flatness
(proxy for whistle/click/burst-pulse — flatness near 0 = tonal/whistle,
flatness near 1 = noise/broadband click).


In [ ]:
diff_coords = diffusion_map_embedding(W, n_components=3, t=5)
tsne = TSNE(n_components=2, perplexity=min(30, max(5, X.shape[0] // 5)),
            init="pca", random_state=0)
tsne_coords = tsne.fit_transform(Xz)

cl_best = clusterings[best_k_data]
cl_k3 = clusterings[3]  # biological reference

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
import matplotlib as mpl
cmap_disc = mpl.colormaps.get_cmap("tab10").resampled(max(best_k_data, 3))

# (0,0) diffusion map colored by k* clusters
ax = axes[0, 0]
for c in sorted(set(cl_best)):
    m = cl_best == c
    ax.scatter(diff_coords[m, 0], diff_coords[m, 1], s=20, alpha=0.7,
               color=cmap_disc(c), label=f"cluster {c}")
ax.set_title(f"Diffusion map t=5, colored by k*={best_k_data} clusters")
ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
ax.legend(fontsize=8)

# (0,1) diffusion map colored by spectral flatness
ax = axes[0, 1]
sc = ax.scatter(diff_coords[:, 0], diff_coords[:, 1], c=flatness_mean,
                cmap="plasma", s=20, alpha=0.8)
ax.set_title("Diffusion map, colored by spectral flatness")
ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04, label="flatness")

# (1,0) tsne colored by k* clusters
ax = axes[1, 0]
for c in sorted(set(cl_best)):
    m = cl_best == c
    ax.scatter(tsne_coords[m, 0], tsne_coords[m, 1], s=20, alpha=0.7,
               color=cmap_disc(c), label=f"cluster {c}")
ax.set_title(f"t-SNE, colored by k*={best_k_data} clusters")
ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
ax.legend(fontsize=8)

# (1,1) diffusion map colored by k=3 (biological reference)
ax = axes[1, 1]
cmap_3 = mpl.colormaps.get_cmap("tab10").resampled(3)
for c in sorted(set(cl_k3)):
    m = cl_k3 == c
    ax.scatter(diff_coords[m, 0], diff_coords[m, 1], s=20, alpha=0.7,
               color=cmap_3(c), label=f"cluster {c}")
ax.set_title("Diffusion map, k=3 (biological reference)")
ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "embeddings.png", dpi=150)
plt.close(fig)
print("saved embeddings.png")


## Validation against spectral-flatness proxy labels

Threshold flatness into 3 acoustic groups using terciles:
  low flatness → whistle/tonal
  mid flatness → burst pulse
  high flatness → click/broadband


In [ ]:
q1, q2 = np.quantile(flatness_mean, [1/3, 2/3])
proxy = np.zeros_like(flatness_mean, dtype=np.int64)
proxy[flatness_mean > q1] = 1
proxy[flatness_mean > q2] = 2
print(f"proxy thresholds: {q1:.4f}, {q2:.4f}")
print("proxy class counts:", np.bincount(proxy))

ari = adjusted_rand_score(proxy, cl_k3)
nmi = normalized_mutual_info_score(proxy, cl_k3)
print(f"k=3 spectral clustering vs flatness proxy: ARI={ari:.4f}, NMI={nmi:.4f}")

ari_best = adjusted_rand_score(proxy, cl_best)
nmi_best = normalized_mutual_info_score(proxy, cl_best)
print(f"k={best_k_data} (best) vs flatness proxy: ARI={ari_best:.4f}, NMI={nmi_best:.4f}")


## Spectrogram grids — visual validation per cluster


In [ ]:
spec = np.load(DATA_DIR / "level3_spectrograms.npz", allow_pickle=True)
S_all = spec["S"].astype(np.float32)
print("S:", S_all.shape)

n_per_row = 8
for cluster_id in sorted(set(cl_k3)):
    idx = np.where(cl_k3 == cluster_id)[0]
    if idx.size == 0: continue
    pick = np.random.default_rng(0).choice(idx, size=min(n_per_row, idx.size), replace=False)
    fig, axes = plt.subplots(1, len(pick), figsize=(2 * len(pick), 2.5))
    if len(pick) == 1:
        axes = [axes]
    for ax, p in zip(axes, pick):
        ax.imshow(S_all[p], origin="lower", aspect="auto", cmap="magma")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"#{p}", fontsize=8)
    fig.suptitle(f"k=3 cluster {cluster_id} — random spectrograms")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"cluster_{cluster_id}_spectrograms.png", dpi=130)
    plt.close(fig)
print("saved cluster spectrogram grids")


## Autoencoder baseline (46 → 32 → 16 → 32 → 46)


In [ ]:
class AE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(46, 32), nn.ReLU(), nn.Linear(32, 16))
        self.dec = nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 46))
    def forward(self, x):
        z = self.enc(x); return self.dec(z), z

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
model = AE().to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)
crit = nn.MSELoss()

Xt = torch.from_numpy(Xz).to(device)
ds = TensorDataset(Xt)
dl = DataLoader(ds, batch_size=32, shuffle=True)

EPOCHS = 200
losses = []
for ep in range(EPOCHS):
    model.train()
    ep_loss = 0.0
    n = 0
    for (xb,) in dl:
        opt.zero_grad()
        rec, _ = model(xb)
        loss = crit(rec, xb)
        loss.backward()
        opt.step()
        ep_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    losses.append(ep_loss / n)
    if (ep + 1) % 20 == 0:
        print(f"  AE epoch {ep+1}/{EPOCHS}: loss={losses[-1]:.4f}")

model.eval()
with torch.no_grad():
    _, Z = model(Xt)
Z = Z.cpu().numpy()
print("AE bottleneck:", Z.shape)

ae_silh = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=0)
    lab = km.fit_predict(Z)
    s = silhouette_score(Z, lab) if len(set(lab)) > 1 else float("nan")
    ae_silh.append(s)
    print(f"  AE+kmeans k={k}: silhouette={s:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_RANGE, silhouettes, marker="o", label="spectral clustering")
ax.plot(K_RANGE, ae_silh, marker="s", label="AE + k-means")
ax.set_xlabel("k"); ax.set_ylabel("silhouette score")
ax.set_title("Level 3 — silhouette score: spectral vs AE")
ax.grid(True, alpha=0.3); ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "silhouette_compare.png", dpi=150)
plt.close(fig)


## Final comparison


In [ ]:
print("=== Level 3 comparison (silhouette per k) ===")
print(f"  k          : {K_RANGE}")
print(f"  spectral   : {[f'{s:.3f}' for s in silhouettes]}")
print(f"  AE+kmeans  : {[f'{s:.3f}' for s in ae_silh]}")
print(f"\nbest spectral k = {best_k_data} (silhouette {max(silhouettes):.4f})")
print(f"k=3 (biological reference) silhouette: {silhouettes[K_RANGE.index(3)]:.4f}")
print(f"\nproxy validation (flatness terciles):")
print(f"  k=3 spectral:    ARI={ari:.4f}, NMI={nmi:.4f}")
print(f"  k={best_k_data} spectral:    ARI={ari_best:.4f}, NMI={nmi_best:.4f}")
print(f"\nFigures saved to {FIG_DIR}")
